# Этот ноутбук содержит в себе визуализации кластеров из объединённого датафрейма АПК + химия

In [141]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils.vis import plot_tsne_similarity, plot_tsne_similarity_plotly
from utils.misc import save_dict_as_json
from sklearn.preprocessing import MinMaxScaler, StandardScaler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [142]:
df1 = pd.read_csv("/home/danilach/mipt-stats/SberTs/Data/apk_nona.csv", index_col=0)
df1.index = pd.to_datetime(df1.index)
df2 = pd.read_csv(
    "/home/danilach/mipt-stats/SberTs/Data/chemicals_nona.csv", index_col=0
)
df2.index = pd.to_datetime(df2.index)

In [143]:
df = pd.concat([df1[24:], df2[:-1]], axis=1)
print(len(df.columns))

51


# T-SNE

Посмотрим на кластеризацию методом t-sne рядов, и на то каким кластерам принадлежат показатели. Посмотри на ра различные метрики расстояния

Метрики расстояния

In [144]:
pdist_list = [
    "l2",
    "correlation",
    "cosine",
]

Словари групп
- `apk_chem_groups` - группы по датасетам (АПК и химия)
- `Dastan groups` - группы Дастана и добавленная химия разбитая по визуальному приницпу

In [145]:
apk_chem_groups = {}
for col in df1.columns:
    apk_chem_groups[col] = "apk"
for col in df2.columns:
    apk_chem_groups[col] = "chem"

dastan_groups = {   
    "Кукуруза": "Grains & Seeds",
    "Пшеница 1-го класса": "Grains & Seeds",
    "Пшеница 3-го класса": "Grains & Seeds",
    "Пшеница 4-го класса": "Grains & Seeds",
    "Пшеница 5-го класса": "Grains & Seeds",
    "Пшеница 12,5% FOB Ново, $/т": "Grains & Seeds",
    "Подсолнечник": "Grains & Seeds",
    "Соя": "Grains & Seeds",
    "Рапс, руб./т": "Grains & Seeds",
    # Oils & Meals
    "Подсолнечное масло наливом (мировые цены)": "Oils & Meals",
    "Бутилированное подсолнечное масло (рафинированное)": "Oils & Meals",
    "Подсолнечное масло (наливом) не бутилированное, нерафинированное ": "Oils & Meals",
    "Подсолнечный шрот ": "Oils & Meals",
    "Соевое масло": "Oils & Meals",
    "Соевый шрот": "Oils & Meals",
    "Рапсовое масло EU, $/т": "Oils & Meals",
    "Кокосовое масло (USD) ": "Oils & Meals",
    "Кокосовое масло ": "Oils & Meals",
    "Пальмовое масло (USD)": "Oils & Meals",
    "Пальмовое масло ": "Oils & Meals",
    # Livestock
    "Молоко сырое": "Livestock & Meat",
    "Мясо птицы бройлеров в живом весе ": "Livestock & Meat",
    "Мясо крупного рогатого скота в живом весе ": "Livestock & Meat",
    "Свинина в живом весе": "Livestock & Meat",
    "Баранина в живом весе": "Livestock & Meat",
    "Баранина в убойном весе ": "Livestock & Meat",
    "Яйцо товарное": "Livestock & Meat",
    # Processed meat
    "Колбасы сырокопченые": "Processed Meat",
    "Колбасы вареные": "Processed Meat",
    # Fish
    "Минтай б/г, Владивосток руб./кг": "Fish",
    "Минтай б/г, Китай C&F $/т": "Fish",
    "Сельдь н/р, Владивосток руб./кг": "Fish",
    # Vegetables
    "Огурцы тепличные, руб./кг": "Vegetables",
    "Томаты тепличные, руб./кг": "Vegetables",
    # Other
    "Сахар (средняя цена по России)": "Sugar & Flour",
    "Мука пшеничная, руб./т": "Sugar & Flour",
    # Stimuli crops
    "Какао-бобы (USD)": "Stimuli Crops",
    "Какао-бобы": "Stimuli Crops",
    "Табак (USD)": "Stimuli Crops",
    "Табак": "Stimuli Crops",
}

visual_chem = {}

## apk_chem_groups

In [146]:
color_dict = {
    key: "red" if value == "apk" else "blue" for key, value in apk_chem_groups.items()
}

In [147]:
for metric in pdist_list:
    plot_tsne_similarity_plotly(
        df=df,
        metric=metric,
        color_dict=color_dict,
        perplexity=5,
        scaler=StandardScaler,
        title=metric,
        legend=False,
    )

## dastan_groups

In [148]:
chem_dict = {key: "dip" if i <= 5 else "bump" for i, key in enumerate(df2.columns)}
dastan_groups.update(chem_dict)

In [149]:
colors = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "violet",
    "olive",
    "cyan",
    "black",
    "yellow"
]
unique_values = list(set(dastan_groups.values()))
value_to_color = {value: colors[i] for i, value in enumerate(unique_values)}
color_dict = {key: value_to_color[value] for key, value in dastan_groups.items()}

In [150]:
for metric in pdist_list:
    plot_tsne_similarity_plotly(
        df=df,
        metric=metric,
        color_dict=color_dict,
        perplexity=5,
        scaler=StandardScaler,
        group_dict=dastan_groups,
        title=metric,
        legend=False,
    )

## V_1 groups

In [151]:
df = pd.read_csv(
    "/home/danilach/mipt-stats/SberTs/Data/comb_df_v1.csv", index_col=0
)
df.index = pd.to_datetime(df.index)

In [152]:
import json

path = "/home/danilach/mipt-stats/SberTs/clusters/v1.json"
with open(path, "r") as json_file:
    groups = json.load(json_file)

In [ ]:
unique_values = list(set(groups.values()))
value_to_color = {value: colors[i] for i, value in enumerate(unique_values)}
color_dict = {key: value_to_color[value] for key, value in groups.items()}

In [154]:
for metric in pdist_list:
    plot_tsne_similarity_plotly(
        df=df,
        metric=metric, 
        color_dict=color_dict,
        perplexity=5,
        scaler=MinMaxScaler,
        group_dict=groups,
        title=metric,
        legend=False,
    )

Несколько наблюдений.

`Fish`: оба Минтая хорошо лежат везде, сельдь только на **cosine**

`Пшеница 12,5% FOB Ново, руб/т` выбивается из кластера зерна и семян. Стоит помнить что это пшеница не российская и цена была получена доиножением на курс $

`Рапсовое масло` с натяжкой в oils  

`Сахар` без кластера. Но не на картинках, он в данном случае без кластера так как идейно он больше никуда не подходит